# needle-fp4: which matmul breaks 4-bit training, and what 4-bit weights save

## 1. Title and members

**CMU 10-414/714 Deep Learning Systems, final project. Group 59 (slot B6).**
Shreya Verma (shreyave), Anshuman Mishra (anshm), Nidhi Prakash (nidhipra).

This notebook is our report and our code. **To reproduce:** Runtime > Run all, with or without a GPU (a GPU adds the
CUDA tests and benchmark). Each experiment cell ends with a one-line **Verdict**; the output is zipped as
`fp4_results.zip`. For a private repository, add a `GITHUB_TOKEN` secret first (see the setup cell).

*AI-assisted: written with Claude (Anthropic) via Claude Code; reviewed and tested by the team (see section 8).*

## 2. Introduction

New GPUs multiply 4-bit floats (NVIDIA's NVFP4): one eighth of FP32's memory, so faster batch-1 inference, but only
15 distinct values, so every operand is rounded hard. A linear layer runs three matmuls in training: **Fprop**
`Y = XW`, **Dgrad** `dX = dY Wᵀ` and **Wgrad** `dW = Xᵀ dY`. Our hypothesis: the gradient matmuls are the fragile
ones, because round-to-nearest on gradients is biased, and **stochastic rounding (SR)**, which rounds up with
probability equal to the fractional distance, removes that bias. We built NVFP4 into needle, our homework framework
(no PyTorch, TensorFlow, JAX or cuBLAS), from a NumPy reference down to CUDA kernels, and found:

* **Round-to-nearest gradients are what break 4-bit training, and stochastic rounding fixes most of it.** On a
  CIFAR-10 MLP trained for 20 epochs (5 seeds), Dgrad with round-to-nearest loses 4.5
  accuracy points against FP32 and Dgrad + SR only 0.4. Rounding only dY costs
  4.5 points while rounding only W costs 0.2: it is the gradient,
  not the weight. With all three matmuls in 4 bits, round-to-nearest collapses (12.4% vs
  52.3%); SR prevents the collapse but still ends 6.1 points behind,
  a gap that only appears with long training.
* **Real 4-bit weights are 7x smaller** (97.42% vs 97.54% MNIST accuracy), and our CUDA kernel multiplies
  directly from them.
* **Stretch goal:** NVFP4 stores weights with about 0.8 to 0.9 dB less error than MXFP4, for 0.25 more bits per weight.

## 3. Background and related work

**The formats.** E2M1 has a sign, 2 exponent bits and 1 mantissa bit: 0, 0.5, 1, 1.5, 2, 3, 4, 6 and their
negatives. Because that range is tiny, each small block of numbers along the matmul's reduction axis shares a scale.

| Format | Block | Block scale | Extra scale | Bits per weight |
|---|---|---|---|---|
| NVFP4 | 16 | E4M3 (8-bit float) | FP32 per tensor | 4.5 |
| MXFP4 (OCP MX) | 32 | E8M0 (power of two) | none | 4.25 |

For NVFP4 the tensor scale is amax / (6 · 448) and each block scale is (block amax / 6) / tensor scale rounded to
E4M3. Every operand is quantized along its own reduction axis, so each of the three matmuls quantizes different axes.

**4-bit training.** NVIDIA pretrained a 12B-parameter model on 10T tokens in NVFP4, keeping a few layers in higher
precision and rounding gradients stochastically [1]. Chmiel et al. [2] train fully in FP4 and argue that gradients
need SR, plus higher precision once gradient noise dominates. Tseng et al. [3] and Quartet [4] train in MXFP4 with SR.
DeepSeek-V3 [5] showed fine-grained block scaling (FP8) at scale.

**Where we fit.** Those works train large models with all the tricks at once; we isolate each matmul and each operand,
with and without SR, at small scale, in a framework we built ourselves.

## 4. Software architecture and design

One rule drives the design: **a NumPy reference defines NVFP4 exactly, and every other layer must match it bit for
bit.**

```
           NumPy reference (fp4/reference.py): quantize, pack, dequantize, GEMV order
              |  bit-exact tests          |  bit-exact tests              |  bit-exact tests
   NDArray fake quant  --->  Autograd Op + QuantConfig        Packed NVFP4 + kernels (CPU, CUDA)
   (fp4/quantize.py)         (ops/ops_fp4.py, fp4/config.py)   (fp4/packed.py, src/nvfp4.h)
                                      |                                    |
                          nn.QuantizedLinear (training)          4-bit inference (GEMV)
```

| Piece | Files | What it does |
|---|---|---|
| Reference | `python/needle/fp4/reference.py` | NumPy quantize, pack, dequantize, and a GEMV that copies the GPU's summation order |
| Fake quant | `python/needle/fp4/quantize.py`, backend `floor`/`abs` | quantize-dequantize on any needle device |
| Autograd | `python/needle/ops/ops_fp4.py`, `python/needle/fp4/config.py` | `FP4Matmul` Op; `QuantConfig` says which of the three matmuls are 4-bit, whether SR is on, and how W is quantized (`weights`: per matmul, 16x16 tiles, or SR re-quantization) |
| Layer | `python/needle/nn/nn_fp4.py` | `nn.QuantizedLinear`, a drop-in for `nn.Linear` |
| Real storage | `python/needle/fp4/packed.py`, `src/nvfp4.h`, both backends | `NVFP4Matrix`: packed codes, E4M3 scales, tensor scale; quantize+pack and GEMV kernels |
| Stretch | `python/needle/fp4/formats.py` | NVFP4 vs MXFP4 weight quantization |
| Experiments | `apps/`, `benchmarks/` | training grid, gradient error, packed inference, GEMV benchmark |

**Three matmuls, three axes.** With W stored as (in, out), each operand is quantized along its own matmul's reduction
axis: X and W along `in` for Fprop, dY and W along `out` for Dgrad, X and dY along the batch for Wgrad. Backward is a
straight-through estimator; SR applies only to dY.

**Fake vs real 4-bit.** Training rounds values to NVFP4 and multiplies in FP32: the exact accuracy effect, no saving.
The tests show it matches the reference bit for bit on every device, so the multi-seed grids ran on CPU. Inference
uses real storage: two codes per byte, one E4M3 scale byte per 16 values, multiplied directly by our kernels.

**Bit-exact kernels.** Float32 operations follow the reference's order (`_rn` intrinsics on the GPU,
`-ffp-contract=off` on the CPU). Quantize+pack reduces each block's amax with a half-warp `__shfl_xor_sync` and pairs
codes with `__shfl_down_sync`; the GEMV gives each warp four rows, reuses each float4-loaded block of x for all four,
and sums lanes in a butterfly order the reference reproduces.

In [ ]:
# Which branch of the repository to run: "main", or a dev/<name>-<topic> branch to try unmerged work.
BRANCH = "main"

In [ ]:
import base64, os, subprocess, sys

# Where to work: Google Drive on Colab (results survive the session), the current folder anywhere else.
try:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/10714'
except ImportError:
    userdata, WORK = None, os.getcwd()

# Private repo: create a GitHub token with read access, then in Colab click the key icon (Secrets), add a
# secret named GITHUB_TOKEN and allow notebook access (elsewhere: set the GITHUB_TOKEN environment variable).
# The token is passed to git for each command and never saved.
try:
    token = userdata.get('GITHUB_TOKEN') if userdata else os.environ.get('GITHUB_TOKEN')
except Exception:
    token = None
basic = base64.b64encode(f"x-access-token:{token}".encode()).decode() if token else ""
AUTH = f"-c http.extraheader='Authorization: Basic {basic}'" if token else ""

if not os.path.exists('python/needle/fp4'):        # not already inside the repository
    os.makedirs(WORK, exist_ok=True)
    os.chdir(WORK)
    if not os.path.exists('fp4-needle'):
        !git {AUTH} clone -q https://github.com/shreya-v7/fp4-needle.git
    os.chdir('fp4-needle')
# Match GitHub exactly (Colab autosaves this notebook into the Drive copy, which would block an update).
# Only tracked files are reset; results/, data/ and build/ are git-ignored and kept.
!git {AUTH} fetch -q origin && git checkout -q -f {BRANCH} && git reset -q --hard origin/{BRANCH}
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
remote = subprocess.check_output(['git', 'rev-parse', f'origin/{BRANCH}'], text=True).strip()
assert head == remote, f"Checkout is at {head[:7]} but origin/{BRANCH} is {remote[:7]}; see the git output above."
print(f"Running branch {BRANCH} at commit {head[:7]} in {os.getcwd()}")

In [ ]:
# Build needle's C++ (CPU) and, when a GPU and CUDA are present, CUDA backends.
!pip3 install -q pybind11
!make clean > /dev/null && make 2>&1 | tail -2
os.environ['PYTHONPATH'] = './python'    # for the scripts run below with !python3
os.environ['NEEDLE_BACKEND'] = 'nd'
sys.path += ['python', 'apps']
import csv
from IPython.display import Image, Markdown, display
import needle as ndl

DEVICE = 'cuda' if ndl.cuda().enabled() else 'cpu'
GPU = DEVICE == 'cuda'
R = 'results/colab'                      # everything this run produces goes here
os.makedirs(R, exist_ok=True)
if GPU:
    !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
    print('Running on the GPU.')
else:
    print('No GPU: running on the CPU. The CUDA tests and the GPU benchmark are skipped; everything else runs.')

## 5. Evaluation

### 5.1 Correctness

`tests/fp4` checks every piece against the reference: E2M1 and E4M3 rounding at every grid point and tie, packing
layout, padding, SR being unbiased, the straight-through gradient, all three matmul placements, the quantize+pack,
dequantize and GEMV kernels on CPU and CUDA, and the MXFP4 quantizer. CI runs the CPU half on every pull request.

In [ ]:
!python3 -m pytest tests/fp4 -q 2>&1 | tee {R}/tests.txt | tail -2
last = open(f'{R}/tests.txt').read().strip().splitlines()[-1]
ok = 'failed' not in last and 'error' not in last
print('\nVerdict:', 'all tests pass.' if ok else 'SOME TESTS FAILED (see above).',
      '' if GPU else 'The skipped tests are the CUDA ones; they run when a GPU is present.')

### 5.2 Which matmul breaks training?

An MLP (input, 256, 256, 10) of `QuantizedLinear` layers, SGD (lr 0.05, momentum 0.9), batch 128, 3 epochs; every
configuration starts from the same weights and sees the same batches.

**CIFAR-10** (all 50,000 training images, 4 seeds, mean ± std):

| FP4 matmuls | test accuracy (%) | train loss (last 100 steps) |
|---|---|---|
| FP32 | 43.96 ± 0.45 | 1.584 |
| Fprop | 43.71 ± 0.84 | 1.590 |
| **Dgrad** | 36.41 ± 3.22 | 1.716 |
| Dgrad + SR | 43.53 ± 0.56 | 1.592 |
| Wgrad | 44.40 ± 0.81 | 1.587 |
| Wgrad + SR | 44.11 ± 0.14 | 1.586 |
| Fprop + Dgrad | 38.50 ± 1.82 | 1.737 |
| Fprop + Dgrad + SR | 43.94 ± 0.63 | 1.604 |
| **All three** | 23.34 ± 3.99 | 2.766 |
| All three + SR | 43.77 ± 0.31 | 1.612 |

Fprop alone and Wgrad alone are harmless; **Dgrad with round-to-nearest costs 7.5 points on
every seed**, all three together far more, and SR closes every gap at this length. **MNIST** is too easy to separate
the configurations (all within 0.4 points; `docs/results/cpu_sgd/`).

In [ ]:
display(Image('docs/results/cpu_cifar_full/cifar_ablation.png'))
display(Image('docs/results/cpu_cifar_full/cifar_grid.png'))
display(Image('docs/results/cpu_cifar_weights/cifar_weights.png'))

### 5.3 What 4-bit rounding does to the gradients

`apps/fp4_grad_error.py` compares each configuration's weight gradients with exact ones on a fixed batch (first two
layers, CIFAR-10, after one epoch). Round-to-nearest errors are all **bias**; for SR, 32 draws split the error into
bias and **noise**, which SGD averages away.

| configuration | error (%) | bias (%) | noise (%) | cosine with exact gradient |
|---|---|---|---|---|
| Fprop, round to nearest | 35.3 | 35.3 | 0.0 | 0.952 |
| Dgrad, round to nearest | 15.9 | 15.9 | 0.0 | 0.989 |
| Dgrad + SR | 14.0 | 11.2 | 10.4 | 0.990 |
| Dgrad, only dY rounded, to nearest | 11.3 | 11.3 | 0.0 | 0.995 |
| Dgrad, only dY rounded, SR | 8.9 | 2.4 | 10.1 | 0.996 |
| Dgrad, only W rounded | 11.0 | 11.0 | 0.0 | 0.994 |

**SR on dY works as theory says**: it turns most of dY's bias into noise. But **error size alone does not predict
harm**: rounding W leaves as much bias as rounding dY, and Fprop's error is the largest, yet Fprop is harmless. Only
training can tell which rounding hurts (sections 5.4 and 5.5). Other checkpoints agree (`docs/results/grad_error_cifar/`).

### 5.4 Is it the gradient rounding or the weights?

Dgrad re-quantizes W along a different axis than Fprop, so the backward pass uses a slightly different weight. NVIDIA
says "errors induced from inconsistent weights materialize in the activation gradients" and uses one scale per 16x16
tile [1]; Thinking Machines re-quantizes the forward weight with SR [8]. Both are a `QuantConfig.weights` option here
(all-FP4, CIFAR-10, 3 epochs, 4 seeds):

| gradient dY rounding | weight quantization for Fprop and Dgrad | from | test accuracy (%) | train loss |
|---|---|---|---|---|
| FP32 |  |  | 43.96 ± 0.45 | 1.584 |
| round to nearest | 1x16, separately per matmul | this project | 23.34 ± 3.99 | 2.766 |
| round to nearest | 16x16 tiles, shared | NVIDIA [1] | 19.80 ± 2.99 | 2.792 |
| round to nearest | forward W re-quantized with SR | Thinking Machines [8] | 20.82 ± 5.34 | 2.477 |
| stochastic | 1x16, separately per matmul | this project | 43.77 ± 0.31 | 1.612 |
| stochastic | 16x16 tiles, shared | NVIDIA [1] | 43.11 ± 0.45 | 1.609 |
| stochastic | forward W re-quantized with SR | Thinking Machines [8] | 42.99 ± 0.72 | 1.617 |

Consistent weights do not rescue round-to-nearest (19.8% and
20.8% vs 23.3%); SR on dY rescues all three schemes, and the
consistent ones even cost a little (coarser tiles, extra SR noise). At billions of parameters and trillions of
tokens weight inconsistencies have longer to add up; here the bias in dY dominates.

### 5.5 Stress tests: 20 epochs, 5 seeds, isolating dY and W (pre-registered)

Does round-to-nearest fail, or only learn more slowly? Every key configuration trained for 20 epochs on all of
CIFAR-10 with 5 seeds. The pass/fail criteria were committed before running
([`docs/PREREGISTRATION.md`](../docs/PREREGISTRATION.md)) and the setup audited for bias
([`docs/BIAS_AUDIT.md`](../docs/BIAS_AUDIT.md)). **Metric A** is the mean test accuracy of the last 3 epochs,
compared per seed.

| configuration | metric A (%) | minus FP32, paired (points) | seeds below FP32 |
|---|---|---|---|
| FP32 | 52.28 ± 0.19 |  |  |
| Dgrad, round to nearest | 47.76 ± 0.68 | -4.53 (SE 0.28) | 5/5 |
| Dgrad + SR | 51.90 ± 0.27 | -0.39 (SE 0.08) | 5/5 |
| Dgrad, only dY rounded to nearest | 47.80 ± 1.67 | -4.48 (SE 0.70) | 5/5 |
| Dgrad, only dY rounded, SR | 51.96 ± 0.27 | -0.32 (SE 0.07) | 5/5 |
| Dgrad, only W rounded | 52.09 ± 0.24 | -0.20 (SE 0.08) | 4/5 |
| All three, round to nearest | 12.38 ± 1.90 | -39.90 (SE 0.83) | 5/5 |
| All three + SR | 46.14 ± 1.97 | -6.14 (SE 0.89) | 5/5 |

**Pre-registered verdicts** (`apps/fp4_stats.py`, full output in `docs/results/extended/verdicts.md`):

1. **H1, round-to-nearest gradients hurt and SR fixes them: supported.** SR beats round-to-nearest by
   4.1 points for Dgrad and 33.8
   for all three, in every seed.
2. **H1b, SR recovers FP32 within 1 point: not supported.** It does for Dgrad (-0.4), but with all
   three matmuls in 4 bits SR ends 6.1 points behind. That gap was
   0.7 points at epoch 3 and grows with training, so 3-epoch runs hid it.
3. **H2, it is the rounding of dY, not of W: supported.** Only dY rounded: -4.5 points;
   only W rounded: -0.2; only dY rounded with SR: -0.3.
4. **H3, failure or slower learning: "slower learning" by the pre-registered rule** (the Dgrad gap is
   9.1 points at epoch 3 and 3.4 at epoch 20). Looking at every epoch
   (not pre-registered), epoch 3 was the gap's peak: from epoch 4 on it stays between 2 and 5 points without closing,
   which reads more like a lasting penalty than a delay.

**Does it depend on the settings?** Learning rate (2 seeds per new value, 10 epochs, metric A over epochs 8 to 10):

| setting | fp32 (%) | dgrad (%) | dgrad+sr (%) | dgrad+sr minus dgrad | holds? |
|---|---|---|---|---|---|
| 0.05 | 49.29 | 43.42 | 48.91 | +5.49 (SE 0.23, 5/5 seeds) | yes |

Depth, in hidden layers (3 seeds per new value, 10 epochs):

| setting | fp32 (%) | dgrad (%) | dgrad+sr (%) | dgrad+sr minus dgrad | holds? |
|---|---|---|---|---|---|
| 2 | 49.29 | 43.42 | 48.91 | +5.49 (SE 0.23, 5/5 seeds) | yes |
| 4 | 49.09 | 39.60 | 49.39 | +5.34 (SE 0.76, 2/2 seeds) | yes |

Rounding W biases the gradient as much as rounding dY (section 5.3), yet only dY hurts. One untested difference: W's
rounding error changes only as slowly as W, while dY's is new at every step.

In [ ]:
display(Image('docs/results/extended/verdicts.png'))

### 5.6 Re-running the key result

The five configurations that carry the result, 1 seed: all 50,000 CIFAR-10 images on a GPU, the first 20,000 on a CPU
so the run stays short. It also saves the FP32 model for section 5.8.

In [ ]:
import tarfile, urllib.request
if not os.path.exists('data/cifar-10-batches-py'):
    os.makedirs('data', exist_ok=True)
    urllib.request.urlretrieve('https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz', 'data/cifar-10-python.tar.gz')
    tarfile.open('data/cifar-10-python.tar.gz').extractall('data')

SUBSET = '' if GPU else '--train-subset 20000'   # all 50,000 images on a GPU, the first 20,000 on a CPU
KEY = 'fp32 dgrad dgrad+sr fprop+dgrad+wgrad fprop+dgrad+wgrad+sr'
!python3 apps/fp4_experiment.py --dataset cifar10 --configs all --only {KEY} --device {DEVICE} --epochs 3 --seed 0 {SUBSET} --out {R}/cifar_check.csv --save-weights "{R}/ckpt/cifar_{{config}}_s{{seed}}.npz"

acc = {r['config']: 100 * float(r['accuracy']) for r in csv.DictReader(open(f'{R}/cifar_check.csv')) if r['split'] == 'test'}
print(f"\nFP32: {acc['fp32']:.1f}%")
for c, name in [('dgrad', 'Dgrad only'), ('fprop+dgrad+wgrad', 'All three')]:
    print(f"{name:11s} round to nearest {acc[c]:5.1f}% ({acc[c] - acc['fp32']:+.1f})   "
          f"with SR {acc[c + '+sr']:5.1f}% ({acc[c + '+sr'] - acc['fp32']:+.1f})")
holds = all(acc[c] < acc[c + '+sr'] - 1 for c in ('dgrad', 'fprop+dgrad+wgrad'))
print('Verdict:', 'the hypothesis holds on this run: round-to-nearest gradients cost accuracy and SR wins it back.'
      if holds else 'this single run does not separate the configurations clearly; the 4-seed results above are the evidence.')

### 5.7 Real 4-bit weights: memory and GEMV speed

At batch size 1 every layer is a matrix-vector product (GEMV), limited by reading the weights. The benchmark
compares our FP32 GEMV with our NVFP4 GEMV, which reads the packed bytes directly, and the homework matmul. The second
command trains an FP32 MNIST MLP, packs its weights and runs it from the packed form.

In [ ]:
import hashlib, urllib.request

# MNIST files and the MD5 of the copy our homeworks used (identical to the course repo).
MNIST = {"train-images-idx3-ubyte.gz": "0cb5796cd8e7c7bffef03edbdb01fac5",
         "train-labels-idx1-ubyte.gz": "23e0cdb926d142c2d470a3641a19d1ff",
         "t10k-images-idx3-ubyte.gz":  "3d4db374c734e4f92d346e063bf6e41a",
         "t10k-labels-idx1-ubyte.gz":  "0407fd84edb9a00bbad73c377ff95de9"}
os.makedirs("data", exist_ok=True)
for name, md5 in MNIST.items():
    dst = os.path.join("data", name)
    if not os.path.exists(dst):
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/dlsyscourse/hw0/main/data/{name}", dst)
    assert hashlib.md5(open(dst, "rb").read()).hexdigest() == md5, f"{dst} is corrupted; delete it and rerun"
print("MNIST ok")

In [ ]:
# Square matrices, plus (on a GPU) the rectangular MLP weights of a 7B-class LLM: up 11008x4096, down 4096x11008.
SHAPES = '--sizes 1024 2048 4096 8192 --shapes 11008x4096 4096x11008' if GPU else '--sizes 1024 2048 4096'
!python3 benchmarks/bench_gemv.py --device {DEVICE} {SHAPES} --out {R}/bench_gemv.csv
display(Image(f'{R}/bench_gemv.png'))
rows = {r['kernel']: r for r in csv.DictReader(open(f'{R}/bench_gemv.csv')) if r['shape'] == '4096x4096'}
fp32, fp4 = rows['fp32 gemv (ours)'], rows['nvfp4 gemv (ours)']
print(f"\nVerdict at 4096 x 4096: weights {float(fp32['weight_MiB']):.0f} MiB -> {float(fp4['weight_MiB']):.1f} MiB; "
      f"NVFP4 GEMV runs at {float(fp32['time_us']) / float(fp4['time_us']):.2f}x the speed of our FP32 GEMV"
      + (f" ({float(fp4['pct_of_peak']):.0f}% vs {float(fp32['pct_of_peak']):.0f}% of peak bandwidth)." if GPU
         else " (CPU wall-clock; the GPU numbers are the ones that matter)."))

# MNIST MLP run from packed 4-bit weights: memory per layer, test accuracy, batch-1 latency
!python3 apps/fp4_inference.py --device {DEVICE} 2>&1 | tee {R}/inference.txt

Packed weights are 7.0x smaller, and the packed MNIST MLP scores 97.42% against 97.54% (`docs/results/cpu_inference/`).
On our first T4 run our FP32 GEMV reached 84% of peak bandwidth (4x the homework matmul), but the first NVFP4 kernel
ran at 0.59x its speed despite 7x fewer bytes: scalar loads of x, `ldexpf` scale decoding, one long dependency chain.
The version above fixes all three without changing the addition order; on a GPU the verdict line gives its speed.

### 5.8 Stretch goal: MXFP4 vs NVFP4

We quantize every weight of a trained FP32 MLP in each format, then measure the error (SQNR = -10 log10(squared
error / weight power), higher is better) and the test accuracy with the quantized weights (`docs/results/mxfp4/`):

| FP32 model | NVFP4, 4.5 bits: SQNR / accuracy | MXFP4, 4.25 bits: SQNR / accuracy | FP32 accuracy |
|---|---|---|---|
| MNIST MLP | 19.81 dB / 95.08% | 18.88 dB / 94.48% | 94.93% |
| CIFAR-10 MLP | 19.85 dB / 43.90% | 19.04 dB / 43.67% | 44.04% |

NVFP4's 16-value blocks and E4M3 scales give about 0.8 to 0.9 dB less error than MXFP4's 32-value power-of-two
blocks, for 0.25 more bits per weight; on these small models both keep accuracy within 0.5 points. The cell below
repeats the comparison on the CIFAR-10 model trained in section 5.6.

In [ ]:
!python3 apps/fp4_mxfp4.py {R}/ckpt/cifar_fp32_s0.npz --device {DEVICE} --out {R}/mxfp4.md

## 6. Discussion and limitations

The hypothesis holds in its main form and fails in one pre-registered detail. Rounding dY to nearest is what breaks
4-bit training; rounding W and the forward pass are harmless; SR removes most of the damage. But SR does not make
all-4-bit training match FP32 over 20 epochs: a gap of several points opens after the first few. Dgrad + SR with
the other matmuls in FP32 does match FP32; which of the other two matmuls opens the all-4-bit gap we did not test.

* **Scale.** needle runs at about 30 GFLOP/s, so we train small MLPs for at most 20 epochs, with SGD only.
* **Fake quantization.** Training measures accuracy, not speed; only inference uses real 4-bit storage.
* **Sweeps.** The learning-rate and depth checks use 2 or 3 seeds: the direction of the effect, not its size.
* **SR is not free of bias.** W and X stay rounded to nearest in the backward matmuls (section 5.3).

## 7. References

1. NVIDIA. [Pretraining Large Language Models with NVFP4](https://arxiv.org/abs/2509.25149). 2025.
2. Chmiel, Fishman, Banner, Soudry. [FP4 All the Way: Fully Quantized Training of LLMs](https://arxiv.org/abs/2505.19115). 2025.
3. Tseng, Yu, Park. [Training LLMs with MXFP4](https://arxiv.org/abs/2502.20586). 2025.
4. Castro et al. [Quartet: Native FP4 Training Can Be Optimal for Large Language Models](https://arxiv.org/abs/2505.14669). 2025.
5. DeepSeek-AI. [DeepSeek-V3 Technical Report](https://arxiv.org/abs/2412.19437). 2024.
6. Open Compute Project. OCP Microscaling Formats (MX) Specification v1.0. 2023.
7. Datasets: MNIST (LeCun et al., 1998); CIFAR-10 (Krizhevsky, 2009).
8. Cook and Thinking Machines Lab. [End-to-End 4-Bit Training](https://thinkingmachines.ai/blog/end-to-end-4-bit-training/). 2026.

## 8. AI usage statement

| Item | Detail |
|---|---|
| Tool | Claude Code (Anthropic), cloud sessions |
| Model and version | TODO (team): the model name and version shown in our Claude Code session |
| Used for | writing code, tests and experiment scripts; debugging (CI, Colab, data paths); finding related work; drafting this report and the README |
| Not used for | the homework needle code we started from, choosing the project, final judgment on claims |

**Affected files.** Every AI-assisted file or edited section starts with the line "AI-assisted: written with Claude
(Anthropic) via Claude Code; reviewed and tested by the team.": all of `python/needle/fp4/`, `ops/ops_fp4.py`,
`nn/nn_fp4.py`; the NVFP4, `floor` and `abs` sections of both backends and `ndarray.py`; `src/nvfp4.h`; the FP4 apps
in `apps/`; `benchmarks/bench_gemv.py`; `tests/fp4/`; this notebook; `.github/workflows/ci.yml`; the
`-ffp-contract=off` line in `CMakeLists.txt`.

**How we verified it.** (1) A NumPy reference written from the NVFP4 spec; every kernel and op is tested bit for bit
against it. (2) CI runs the tests on every pull request, and nothing reaches main without one. (3) Every number in
this report comes from a run we can repeat with the commands below. (4) Claims were checked against the cited papers.
(5) Each of us read and can explain the code in the parts we present.

## 9. Code and reproduction

Code: our private GitHub repository `shreya-v7/fp4-needle` (link shared with the course staff). Run all reproduces
sections 5.1, 5.6, 5.7 and 5.8. The 20-epoch study of section 5.5 runs with `bash scripts/run_extended.sh` (about
6 hours on 4 CPU cores; resumable) and is scored by `python3 apps/fp4_stats.py docs/results/extended --out
docs/results/extended/verdicts.md`. The other multi-seed CPU results come from:

```bash
make lib && export PYTHONPATH=python
for s in 0 1 2 3; do
  python3 apps/fp4_experiment.py --dataset cifar10 --configs all --epochs 3 --seed $s --out results/cifar_full_seed$s.csv
  python3 apps/fp4_experiment.py --dataset cifar10 --configs weights --epochs 3 --seed $s --out results/cifar_weights_seed$s.csv
done
python3 apps/plot_fp4.py results/cifar_full_seed*.csv --out-dir results/cifar_full --title "CIFAR-10 MLP" --prefix cifar
python3 apps/fp4_grad_error.py --dataset cifar10 --checkpoints 0 391 1173 --out results/grad_error_cifar.csv
for s in 0 1 2; do python3 apps/fp4_experiment.py --configs all --epochs 3 --seed $s --out results/mnist_seed$s.csv; done
python3 apps/fp4_experiment.py --configs grid --only fp32 --epochs 3 --out results/mnist_fp32.csv --save-weights "results/ckpt/mlp_{config}_s{seed}.npz"
python3 apps/fp4_mxfp4.py results/ckpt/mlp_fp32_s0.npz --out results/mxfp4_mnist.md
```

The last cell zips this run's output; download `fp4_results.zip` from the Files panel (on Colab it is also in
`MyDrive/10714/fp4-needle/` on Drive).

In [ ]:
import zipfile
with zipfile.ZipFile('fp4_results.zip', 'w', zipfile.ZIP_DEFLATED) as z:   # model checkpoints left out
    for folder, _, files in os.walk(R):
        for f in files:
            if not f.endswith('.npz'):
                z.write(os.path.join(folder, f))
print('Created', os.path.abspath('fp4_results.zip'), os.path.getsize('fp4_results.zip') // 1024, 'KB: download it and send it.')